# Results

This notebook recreates the simulation results in the CEP paper.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "Simulator"))
using Simulator

## Three Scenarios

We consider a scenario with a constant ocean current and no disturbances (CC), a scenario with a variable ocean current and no disturbances (VC), and a scenario with a variable ocean current and sinusoidal disturbances (VCD).

In [ ]:
sim_cc = default_simulation()
sim_vc = default_simulation(; V_c=default_varying_current)
sim_vcd = default_simulation(; V_c=default_varying_current, μ=default_perturbation)


## Plotting

We want a figure with four subplots: path-following errors, velocity errors, angular velocity of the cables, and a North-East plot showcasing the configuration of the system at various time instances.

In [ ]:
using DifferentialEquations
using Plots, ColorSchemes

# Prepare four subplots
path_err_plot = plot(title="Path-following errors", xlabel="Time (s)", ylabel="Error (m)", legend=:bottomright)
vel_err_plot = plot(title="Velocity errors", xlabel="Time (s)", ylabel="Error (m/s)", legend=:bottomright)
ang_vel_plot = plot(title="Angular velocities", xlabel="Time (s)", ylabel="Angular velocity (rad/s)", legend=:bottomright)
traj_plot = plot(title="Trajectory", xlabel="East (m)", ylabel="North (m)", legend=:topleft, aspect_ratio=:equal)

# Plot the desired path
arg = range(0.0, stop=2π, length=100)
p_path = hcat(default_path.(arg)...)
plot!(traj_plot, p_path[2, :], p_path[1, :], label="Path", color=:black)

# Plotting function for each simulation
function simulate_and_plot(sim, color, label)
    # Simulate the system
    x0 = vectorize(default_initial_state())
    prob = ODEProblem(closed_loop_ode, x0, (0.0, 200.0), sim)
    sol = solve(prob, Tsit5(), saveat=0.25)

    # Extract closed-loop states
    cls = [get_closed_loop_state(u_i, sim) for u_i in sol.u]
    e_x = [cls_i.e_x for cls_i in cls]
    e_y = [cls_i.e_y for cls_i in cls]
    v_x = [cls_i.v_x for cls_i in cls]
    v_y = [cls_i.v_y for cls_i in cls]
    θ_dot = [sum(cls_i.θ_dot) / length(cls_i.θ_dot) for cls_i in cls]  # Average angular velocity of the cable

    # Plot errors and angular velocity
    plot!(path_err_plot, sol.t, e_x, label="$label, e_x", color=color)
    plot!(path_err_plot, sol.t, e_y, label="$label, e_y", color=color, linestyle=:dash)
    plot!(vel_err_plot, sol.t, v_x, label="$label, v_x", color=color)
    plot!(vel_err_plot, sol.t, v_y, label="$label, v_y", color=color, linestyle=:dash)
    plot!(ang_vel_plot, sol.t, θ_dot, label="$label", color=color)

    # Plot the configuration at t = 0, 50, 150, 200
    t_select = [0.0, 50.0, 150.0, 200.0]
    pls = [get_plotting_state(sol(t_i), sim) for t_i in t_select]
    for pls_i in pls
        plot_towed_system!(traj_plot, pls_i, default_asv_dimensions(), color)
    end
end

# Simulate and plot for each scenario
colors = ColorSchemes.Dark2_8
simulate_and_plot(sim_cc, colors[1], "CC")
simulate_and_plot(sim_vc, colors[2], "VC")
simulate_and_plot(sim_vcd, colors[3], "VCD")

# Display the plots
plt = plot(path_err_plot, vel_err_plot, ang_vel_plot, traj_plot, layout=(2, 2), size=(1200, 800))